# TRIBE counterfactual re-run on 18-clip benchmark

Compute `accessibility_gap`, `description_gain`, and `alignment_cosine` per clip using the TRIBE v2 counterfactual proxy:

- `P_AV`  = full video prediction (audio + visual)
- `P_A`   = audio-only prediction (extracted wav)
- `P_AD`  = AD text-only prediction (TRIBE language path)

Then:

- `accessibility_gap = 1 - cos(P_AV, P_A)`  -- what audio alone leaves out
- `description_gain  = cos(P_AV, P_AD) - cos(P_AV, P_A)`  -- AD's measurable value-add
- `alignment_cosine  = cos(P_AV, P_AV+AD_overlay)`  -- legacy proxy (kept for comparison)

Outputs per-clip CSV + per-feature AUC vs existing failure targets.

## Setup

**On Colab:** mount the SceneTwin repo (or upload as zip). The notebook assumes the working directory is the repo root.

In [ ]:
# Colab-only: mount Google Drive or clone repo. Skip on local.
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust this path to wherever you keep SceneTwin in Drive
    REPO = '/content/drive/MyDrive/SceneTwin'
    os.chdir(REPO)
else:
    REPO = os.getcwd()
print('Repo root:', REPO)

In [ ]:
# Install TRIBE v2 dependencies if missing.
# If running in the existing workspace/tribev2/.venv, this is a no-op.
%pip install -q einops exca pyyaml moviepy huggingface_hub gtts langdetect spacy soundfile Levenshtein scikit-learn pandas numpy
%pip install -q 'torch>=2.5.1,<2.7' 'torchvision>=0.20,<0.22'
%pip install -q neuralset==0.0.2 neuraltrain==0.0.2 x_transformers==1.27.20
# Install the local tribev2 package
%pip install -q -e workspace/tribev2

In [ ]:
import sys
from pathlib import Path
ROOT = Path(REPO if IN_COLAB else os.getcwd()).resolve()
sys.path.insert(0, str(ROOT / 'demo'))
import live_pipeline as lp
print('live_pipeline loaded from', lp.__file__)
ok, msg = lp._load_tribe()
print('TRIBE load:', ok, msg)
assert ok, 'TRIBE failed to load -- fix deps before continuing'

In [ ]:
import pandas as pd

FORECAST_CSV = ROOT / 'output' / 'scenetwin_timing_20clip' / 'tribe_native' / 'tribe_failure_forecast.csv'
CLIPS_DIR    = ROOT / 'workspace' / 'vatex_clips'
OUT_DIR      = ROOT / 'cursor' / 'research' / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

forecast = pd.read_csv(FORECAST_CSV)
roster = forecast[['clip_idx', 'video_id', 'category', 'duration_s',
                   'tier3_va11y_text',
                   'all4_fail', 'low_tier3_margin',
                   'tier2_tier1_inversion', 'quality_risk_fail',
                   'target', 'risk_score']].copy()
print('Clips to process:', len(roster))
roster.head(3)

In [ ]:
def find_clip_video(clip_idx: int) -> Path | None:
    for ext in ('.mp4', '.mkv', '.webm'):
        p = CLIPS_DIR / f'clip_{clip_idx:02d}{ext}'
        if p.exists():
            return p
    return None

def run_one(clip_idx: int, ad_text: str) -> dict:
    video = find_clip_video(clip_idx)
    if video is None:
        return {'clip_idx': clip_idx, 'error': 'no video found'}
    ok, msg, payload = lp.stage_tribe_proxy(str(video), ad_text)
    out = {'clip_idx': clip_idx, 'video': video.name, 'ok': ok, 'msg': msg}
    out.update(payload)
    return out

# Smoke test: run on one clip first to catch issues before all 18
smoke = run_one(int(roster.iloc[0]['clip_idx']), str(roster.iloc[0]['tier3_va11y_text']))
smoke

In [ ]:
# Run all 18 clips
import time
rows = []
t0 = time.time()
for i, r in roster.iterrows():
    cidx = int(r['clip_idx'])
    print(f"[{i+1}/{len(roster)}] clip_{cidx:02d}  {r['video_id']}  ({r['category']})")
    rows.append(run_one(cidx, str(r['tier3_va11y_text'])))
    elapsed = time.time() - t0
    print(f'   elapsed: {elapsed:.0f}s  ok={rows[-1].get("ok")}')

print(f'\nDone in {time.time()-t0:.0f}s')

In [ ]:
res = pd.DataFrame(rows)
merged = roster.merge(res, on='clip_idx', how='left')
per_clip_path = OUT_DIR / 'tribe_counterfactual_per_clip.csv'
merged.to_csv(per_clip_path, index=False)
print(f'Per-clip results -> {per_clip_path}')
merged[['clip_idx','video_id','category','ok','accessibility_gap','description_gain','alignment_cosine','audio_only_ok']]

In [ ]:
# AUC for new features vs existing failure targets
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

def auc_oriented(y: np.ndarray, x: np.ndarray):
    if np.unique(y).size < 2:
        return float('nan'), 'n/a'
    a = roc_auc_score(y, x)
    if a < 0.5:
        return roc_auc_score(y, -x), 'low_bad'
    return a, 'high_bad'

new_features = ['accessibility_gap', 'description_gain', 'alignment_cosine']
targets = ['all4_fail', 'low_tier3_margin', 'tier2_tier1_inversion', 'quality_risk_fail']
auc_rows = []
for t in targets:
    if t not in merged.columns:
        continue
    y = merged[t].astype(float).values
    if np.unique(y[~np.isnan(y)]).size < 2:
        continue
    for f in new_features:
        if f not in merged.columns:
            continue
        x = merged[f].astype(float).values
        mask = ~np.isnan(x) & ~np.isnan(y)
        if mask.sum() < 4:
            continue
        a, direction = auc_oriented(y[mask], x[mask])
        try:
            ap = average_precision_score(y[mask], x[mask] if direction == 'high_bad' else -x[mask])
        except Exception:
            ap = float('nan')
        auc_rows.append({
            'target': t, 'feature': f, 'direction': direction,
            'auc_oriented': a, 'average_precision': ap,
            'n': int(mask.sum()), 'positives': int(y[mask].sum()),
        })

auc_df = pd.DataFrame(auc_rows).sort_values(['target', 'auc_oriented'], ascending=[True, False])
auc_path = OUT_DIR / 'tribe_counterfactual_feature_auc.csv'
auc_df.to_csv(auc_path, index=False)
print(f'AUC table -> {auc_path}')
auc_df

In [ ]:
# Compare to existing top forecast features (from output/.../tribe_failure_robustness_feature_auc.csv)
EXISTING_AUC = ROOT / 'output' / 'scenetwin_timing_20clip' / 'tribe_native' / 'tribe_failure_robustness_feature_auc.csv'
existing = pd.read_csv(EXISTING_AUC)

comparison_rows = []
for t in targets:
    # Top existing on the same target family
    sub_e = existing[existing['target'].str.contains(t, na=False)].sort_values('auc_oriented', ascending=False)
    if len(sub_e) == 0:
        continue
    top_e = sub_e.iloc[0]
    sub_n = auc_df[auc_df['target'] == t].sort_values('auc_oriented', ascending=False)
    if len(sub_n) == 0:
        continue
    top_n = sub_n.iloc[0]
    comparison_rows.append({
        'target': t,
        'existing_top_feature': top_e['feature'],
        'existing_top_auc': float(top_e['auc_oriented']),
        'new_top_feature': top_n['feature'],
        'new_top_auc': float(top_n['auc_oriented']),
        'delta': float(top_n['auc_oriented']) - float(top_e['auc_oriented']),
    })
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(OUT_DIR / 'tribe_counterfactual_vs_existing.csv', index=False)
comparison

In [ ]:
# Write markdown summary
lines = ['# TRIBE counterfactual re-run -- 18-clip benchmark', '']
lines += [f'Source: `cursor/research/tribe_counterfactual_colab.ipynb`',
          f'Per-clip: `cursor/research/output/tribe_counterfactual_per_clip.csv`',
          f'Feature AUC: `cursor/research/output/tribe_counterfactual_feature_auc.csv`',
          '']
lines += ['## New feature AUC vs existing forecast targets', '']
for t in targets:
    sub = auc_df[auc_df['target'] == t]
    if sub.empty:
        continue
    lines += [f"### Target: `{t}`  (n={int(sub['n'].iloc[0])}, positives={int(sub['positives'].iloc[0])})", '']
    lines += ['| feature | direction | AUC | AP |', '|---|---|---:|---:|']
    for _, row in sub.iterrows():
        lines.append(f"| `{row['feature']}` | {row['direction']} | {row['auc_oriented']:.3f} | {row['average_precision']:.3f} |")
    lines.append('')

lines += ['## Comparison vs existing top features', '']
lines += ['| target | existing top | existing AUC | new top | new AUC | delta |',
          '|---|---|---:|---|---:|---:|']
for _, row in comparison.iterrows():
    lines.append(f"| {row['target']} | `{row['existing_top_feature']}` | {row['existing_top_auc']:.3f} | `{row['new_top_feature']}` | {row['new_top_auc']:.3f} | {row['delta']:+.3f} |")
lines.append('')

lines += ['## Paper narrative hooks', '',
         '- If `accessibility_gap` AUC >= existing best: "Directly measured neural counterfactual matches heuristic slot scoring."',
         '- If `accessibility_gap` AUC < existing but >= 0.85: "Provides theoretically motivated, complementary signal."',
         '- If `description_gain` correlates with low margin: "AD value-add is the right calibration variable."',
         '']

summary_path = OUT_DIR / 'tribe_counterfactual_summary.md'
summary_path.write_text('\n'.join(lines))
print(f'Summary -> {summary_path}')
print('\n'.join(lines))

## When you're done

Three files end up in `cursor/research/output/`:

- `tribe_counterfactual_per_clip.csv` -- per-clip accessibility_gap, description_gain, alignment_cosine
- `tribe_counterfactual_feature_auc.csv` -- AUC for each new feature against each failure target
- `tribe_counterfactual_vs_existing.csv` -- head-to-head vs existing best features
- `tribe_counterfactual_summary.md` -- paper-ready summary

Pull them back into the repo and I'll fold the numbers into the TRIBE writeup and the calibration-layer story (task #9).